# Example 1 — MCP Tool Calling with Claude

## What you'll learn
- Why LLMs are limited without external tools
- How to start an MCP server and connect to it from a Python client
- How Claude automatically decides when to call a tool and what parameters to pass
- How to route Claude's tool requests to the MCP server and get back a grounded answer

## Before you start
1. Make sure your `.env` file contains `ANTHROPIC_API_KEY=your_key_here`
2. Start the MCP server in a **separate terminal** (keep it running throughout the notebook):
   ```bash
   cd agentic-ai-workshop
   python mcp_server/weather_server.py
   ```
   You should see:
   ```
   🌤️  Weather MCP Server starting at http://127.0.0.1:8000
       SSE endpoint : http://127.0.0.1:8000/sse
   ```
3. Run all cells **top to bottom** in order.

---
## The Big Picture — Why Does an LLM Need Tools?

Think of a **brilliant colleague who has read every book ever written** — but who has been in a coma since their training ended. They know everything about physics, history, cooking, and code, but they genuinely don't know what the weather is *right now*. You wouldn't blame them for that — it's just not the kind of knowledge you can memorise.

An LLM is exactly that colleague. To get live, real-world information, it needs a **tool** — a bridge to the outside world. That's what **MCP (Model Context Protocol)** provides: a standardised way for an LLM to request that a function be run on its behalf, inspect the result, and incorporate it into its reply.

### The Full MCP Loop (what this notebook builds)

```
┌──────────────────────────────────────────────────────────────────────┐
│                     FULL MCP TOOL-CALLING LOOP                       │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────────────────┐   Step 4: list_tools()  ┌────────────────┐ │
│  │   Your Notebook     │ ──────────────────────► │   MCP Server   │ │
│  │   (MCP Client)      │ ◄────────────────────── │  weather_      │ │
│  └─────────────────────┘  tool schemas returned  │  server.py     │ │
│           │                                       │  :8000/sse     │ │
│           │ Step 5: pass schemas to Claude        └───────┬────────┘ │
│           ▼                                               │          │
│  ┌─────────────────────┐                                  │          │
│  │     Claude LLM      │  Decides to call get_weather     │          │
│  │   (Reasoning)       │  → stop_reason = "tool_use"      │          │
│  └─────────────────────┘                                  │          │
│           │                                               │          │
│           │ Step 6: tool_use signal received              │          │
│           ▼                                               │          │
│  ┌─────────────────────┐   call_tool("get_weather")       │          │
│  │   Your Notebook     │ ──────────────────────►          │          │
│  │   (MCP Client)      │ ◄──────────────────────          │          │
│  └─────────────────────┘  {temperature:31, condition:..}  │          │
│           │                                               │          │
│           │ send tool_result back to Claude               │          │
│           ▼                                               │          │
│  ┌─────────────────────┐                                  │          │
│  │     Claude LLM      │  "Mumbai is 31°C and clear."     │          │
│  │   (Grounding)       │ ──────────────────────────────►  │          │
│  └─────────────────────┘                                  │          │
│                                                           │          │
└──────────────────────────────────────────────────────────────────────┘
```

> **Key insight:** The LLM never *runs* the tool itself — it only *requests* a tool call. Your notebook (acting as the **MCP client**) calls the **MCP server**, which executes the function and returns the result. This separation keeps the LLM in charge of *reasoning* while keeping *execution* safe and auditable.

### What is MCP?

**Model Context Protocol (MCP)** is an open standard (released by Anthropic, 2024) that defines how tools are described, discovered, and called across a network. In this notebook we use a real MCP server and client — so you experience the full protocol, not a simulation.

---
## Step 1 — Install Dependencies

In [1]:
# Install all required packages (safe to re-run)
!pip install -r ../requirements.txt -q

---
## Step 2 — Set Up the Anthropic Client

We load your API key from the `.env` file and create a client to communicate with Claude.

In [2]:
import os
import json
from dotenv import load_dotenv
from anthropic import Anthropic

# Load ANTHROPIC_API_KEY from the .env file
load_dotenv()

client = Anthropic()

# Quick check
if os.getenv("ANTHROPIC_API_KEY"):
    print("✅ API key loaded successfully.")
else:
    print("❌ API key NOT found. Check that your .env file exists and contains ANTHROPIC_API_KEY.")

✅ API key loaded successfully.


---
## Step 3 — Ask Claude Without Any Tool (Baseline)

Let's first ask Claude about real-time weather **with no tools attached**.

Claude is a language model — it only knows what it was trained on. It has no access to live data, so watch how it responds.

In [3]:
prompt = "What is the current weather in Mumbai?"

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}]
)

print("Claude without a tool:")
print("-" * 40)
print(response.content[0].text)

Claude without a tool:
----------------------------------------
I don't have the ability to check **real-time weather data**, so I can't tell you the current weather in Mumbai.

To get the latest weather, you can:
- 🌐 Visit **weather.com** or **timeanddate.com**
- 🔍 Search **"Mumbai weather"** on Google
- 📱 Check a weather app like **AccuWeather** or the built-in weather app on your phone

Generally speaking, Mumbai has a **tropical climate** — hot and humid, with a heavy monsoon season (June–September). Would you like to know more about Mumbai's typical climate?


**What to notice:** Claude correctly says it cannot access real-time weather data.
This is the core limitation we'll fix by giving it a tool.

### Why This Is Actually Correct Behaviour

Notice that Claude didn't make something up — it said *"I don't have real-time data."* This is a sign of a well-aligned model: it knows the boundary of its own knowledge and says so honestly.

But for a real application you can't tell your users "go check a weather website yourself." That's where tools come in. The next steps will give Claude the ability to fetch real data and answer with confidence.

> **Hallucination vs. Honest Uncertainty:** Some LLMs, when they don't know something, will confidently invent an answer (this is called *hallucination*). Claude is designed to express uncertainty instead. Tools eliminate this entire class of problems for factual, retrievable information.


---
## Step 4 — Connect to the MCP Server and Discover Tools

Here is where the real MCP begins. Instead of hardcoding a tool schema, we **ask the server** what tools it exposes.

The MCP server you started in the terminal registered `get_weather` as a tool when it launched. Our client calls `list_tools()` over SSE and gets back the full schema — name, description, and input types — without us writing any JSON by hand.

We then convert that schema into the format Claude's API expects, so Claude knows exactly what tools it can call.

```
Notebook (client)  ──── list_tools() ────►  MCP Server (:8000/sse)
                   ◄─── tool schemas ──────  [get_weather, ...]
```

> **Why this matters:** tools are defined *once*, on the server. Any client — a notebook, a web app, an agent framework — discovers and uses them automatically. Add a new tool to the server and all clients pick it up on the next `list_tools()` call.

In [4]:
import asyncio
import json
import nest_asyncio
from fastmcp.client import Client
from fastmcp.client.transports import SSETransport

# nest_asyncio lets us use asyncio.run() inside Jupyter's existing event loop
nest_asyncio.apply()

MCP_SERVER_URL = "http://127.0.0.1:8000/sse"

async def list_mcp_tools():
    """Connect to the MCP server and return its tool list."""
    async with Client(SSETransport(MCP_SERVER_URL)) as client:
        return await client.list_tools()

# ── Discover tools from the running MCP server ──────────────────────────────
print(f"Connecting to MCP server at {MCP_SERVER_URL} ...")
mcp_tools = asyncio.run(list_mcp_tools())

# ── Convert MCP tool format → Anthropic API format ──────────────────────────
# MCP tool:  tool.name, tool.description, tool.inputSchema  (JSON Schema dict)
# Anthropic: {"name": ..., "description": ..., "input_schema": ...}
tools = [
    {
        "name": t.name,
        "description": t.description or "",
        "input_schema": t.inputSchema,
    }
    for t in mcp_tools
]

print(f"✅ Discovered {len(tools)} tool(s) from the MCP server:\n")
for t in tools:
    print(f"  Tool : {t['name']}")
    print(f"  Desc : {t['description']}")
    print(f"  Schema: {json.dumps(t['input_schema'], indent=4)}\n")

Connecting to MCP server at http://127.0.0.1:8000/sse ...
✅ Discovered 1 tool(s) from the MCP server:

  Tool : get_weather
  Desc : Get the current weather for a given city. Returns temperature in Celsius and a short weather condition.
  Schema: {
    "additionalProperties": false,
    "properties": {
        "city": {
            "type": "string"
        }
    },
    "required": [
        "city"
    ],
    "type": "object"
}



---
## Step 5 — Define an MCP Tool Caller

When Claude signals it wants to call a tool, we need to forward that request to the MCP server rather than running a local Python function.

`call_mcp_tool()` below opens a short-lived session to the server, calls the named tool with Claude's chosen parameters, and returns the result as a JSON string ready to send back to Claude.

```
Notebook (client)  ──── call_tool("get_weather", {"city": "Mumbai"}) ────►  MCP Server
                   ◄─── {"temperature": 31, "condition": "Clear"} ─────────  (executes get_weather)
```

> **The tool code lives on the server, not in this notebook.** If you want to add a new city or change what the tool returns, edit `mcp_server/weather_server.py` and restart the server — no changes needed here.

In [5]:
async def call_mcp_tool(tool_name: str, tool_input: dict) -> str:
    """
    Call a tool on the MCP server.
    Returns the result as a JSON string (ready to pass back to Claude as tool_result content).
    """
    async with Client(SSETransport(MCP_SERVER_URL)) as client:
        result = await client.call_tool(tool_name, tool_input)

    # result.content is a list of MCP content blocks.
    # For simple dict-returning tools FastMCP serialises each value as a TextContent block.
    # We collect all text parts and join them into one JSON string.
    parts = [block.text for block in result.content if hasattr(block, "text")]
    return parts[0] if len(parts) == 1 else json.dumps(parts)


# ── Quick sanity check — call the server directly ───────────────────────────
print("Testing MCP tool call: get_weather(Mumbai)")
test = asyncio.run(call_mcp_tool("get_weather", {"city": "Mumbai"}))
print("Server returned:", test)

print("\nTesting MCP tool call: get_weather(London)")
test2 = asyncio.run(call_mcp_tool("get_weather", {"city": "London"}))
print("Server returned:", test2)

Testing MCP tool call: get_weather(Mumbai)
Server returned: {"temperature":31,"condition":"Clear"}

Testing MCP tool call: get_weather(London)
Server returned: {"temperature":18,"condition":"Cloudy"}


### What Just Happened?

Two real network calls went to your MCP server and came back with weather data. No local Python function was involved — the logic lives entirely in `weather_server.py`.

This is the MCP pattern in action:
- **Server** owns the tool implementation (`get_weather` in `weather_server.py`)
- **Client** (this notebook) discovers and calls tools over the network
- **Claude** only sees the schema — it has no idea whether the tool runs locally, remotely, or across the internet

This separation means you can swap, update, or scale the server without touching the client code at all.

---
## Step 6 — Ask Claude With the Tool (Full MCP Loop)

Now we run the complete three-round exchange. Each round is a distinct API call:

| Round | What we send | What we get back |
|-------|-------------|-----------------|
| **1** | User question + tools (discovered from MCP server) | `stop_reason = "tool_use"` + tool call parameters |
| **2** | *(we forward the call to the MCP server)* | Tool result JSON from the server |
| **3** | Full conversation history + tool result | Final grounded answer from Claude |

> **`stop_reason = "tool_use"`** means Claude wants to call a tool. When you see this, your code takes over: forward the request to the MCP server, collect the result, and send it back. Claude then continues with a grounded answer.

> **`stop_reason = "end_turn"`** means Claude answered directly without needing any tool — you'll see this when you ask a non-weather question in the Try It Yourself section.

In [6]:
prompt = "What is the current weather in Mumbai?"

# ── Round 1: Send question + MCP-discovered tools to Claude ─────────────────
print("[1] Sending question to Claude (tools sourced from MCP server)...")
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=300,
    tools=tools,           # discovered from MCP server in Step 4
    messages=[{"role": "user", "content": prompt}]
)

print(f"    Stop reason: {response.stop_reason}")

if response.stop_reason != "tool_use":
    print("    Claude did not use the tool. Response:", response.content[0].text)
else:
    # ── Round 2: Forward the tool request to the MCP server ──────────────────
    tool_call = next(b for b in response.content if b.type == "tool_use")
    print(f"\n[2] Claude wants to call : '{tool_call.name}'")
    print(f"    With parameters       : {json.dumps(tool_call.input, indent=4)}")

    # Route to MCP server instead of a local function
    print(f"\n[3] Forwarding to MCP server at {MCP_SERVER_URL} ...")
    tool_result = asyncio.run(call_mcp_tool(tool_call.name, tool_call.input))
    print(f"    MCP server returned   : {tool_result}")

    # ── Round 3: Send MCP result back to Claude for the final answer ─────────
    final_response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        tools=tools,
        messages=[
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [{
                    "type":        "tool_result",
                    "tool_use_id": tool_call.id,
                    "content":     tool_result   # JSON string from the MCP server
                }]
            }
        ]
    )

    print("\n[4] Claude's final answer (grounded in MCP server data):")
    print("-" * 50)
    print(final_response.content[0].text)

[1] Sending question to Claude (tools sourced from MCP server)...
    Stop reason: tool_use

[2] Claude wants to call : 'get_weather'
    With parameters       : {
    "city": "Mumbai"
}

[3] Forwarding to MCP server at http://127.0.0.1:8000/sse ...
    MCP server returned   : {"temperature":31,"condition":"Clear"}

[4] Claude's final answer (grounded in MCP server data):
--------------------------------------------------
The current weather in **Mumbai** is:

- 🌡️ **Temperature:** 31°C
- ☀️ **Condition:** Clear

It's a warm and clear day in Mumbai! Make sure to stay hydrated if you're heading out. 😊


---
## Try It Yourself

Work through these challenges in order — each one builds on the last.

### Challenge 1 — Change the city
Edit the `prompt` variable in Step 6 to ask about `Delhi` or `London`, then re-run the cell. Observe that Claude automatically extracts the city name and routes the call to the MCP server.

### Challenge 2 — Add a new city (server-side change)
Open `mcp_server/weather_server.py` and add a new entry to `weather_data`, e.g.:
```python
"Kolkata": {"temperature": 29, "condition": "Partly Cloudy"},
```
Restart the server (`Ctrl+C` then `python mcp_server/weather_server.py`), then ask Claude about Kolkata. Notice you made **zero changes** to the notebook — the new data is picked up automatically.

### Challenge 3 — Ask a non-weather question
Try `prompt = "What is 2 + 2?"` in Step 6. Does Claude call the weather tool? What `stop_reason` do you see? This demonstrates **selective tool use** — Claude is smart enough not to call a tool when it isn't needed.

### Challenge 4 — Ask about an unsupported city
Try `prompt = "What's the weather in Singapore?"`. The server returns `"unknown"` values. Watch how Claude handles this gracefully in its final answer.

### Challenge 5 (Stretch) — Add a second tool to the server
In `weather_server.py`, define a second `@mcp.tool()` function, e.g. `get_air_quality(city: str)`. Restart the server. Re-run Step 4 — you should now see two tools discovered. Then ask Claude: `"What is the weather and air quality in Mumbai?"` and extend the loop in Step 6 to handle multiple tool calls.

> **Hint for Challenge 5:** Claude may return multiple `tool_use` blocks. Iterate over `response.content`, call `call_mcp_tool()` for each one, and send all results back in the same `tool_result` list.

---
## Summary

You just ran a complete, real MCP tool-calling loop — server, client, and LLM all working together.

| Step | What you did | Key concept |
|------|-------------|-------------|
| Step 3 | Asked Claude without tools | LLMs have no live data — honest uncertainty, not hallucination |
| Step 4 | Connected to the MCP server | `list_tools()` discovers tool schemas dynamically — no hardcoding |
| Step 5 | Defined an MCP tool caller | `call_tool()` routes execution to the server over the network |
| Step 6 | Ran the full loop | Claude reasons → MCP server executes → Claude grounds its answer |

### The Three-Party Pattern

```
Claude (LLM)          MCP Client (notebook)        MCP Server
──────────────        ─────────────────────        ───────────────
Sees tool schemas  ←── list_tools() result  ←── registers tools
Signals tool_use   ──► catch & forward      ──► call_tool()
Receives result    ◄── tool_result sent     ◄── returns data
Writes final answer
```

### Key Terms to Remember

- **MCP (Model Context Protocol):** The open standard for tool discovery and remote execution
- **SSE (Server-Sent Events):** The transport protocol our server uses — a lightweight HTTP-based stream
- **`list_tools()`:** How the client learns what tools a server exposes
- **`call_tool()`:** How the client executes a tool on the server
- **`stop_reason = "tool_use"`:** Claude's signal that it wants a tool called
- **`stop_reason = "end_turn"`:** Claude answered directly — no tool needed

---
➡️ **Next:** Open `02_llamaindex_rag.ipynb` to build a RAG pipeline that lets Claude answer questions over your own documents.